# Thu thập dữ liệu "Hiện vật kể chuyện" - Bảo tàng Hồ Chí Minh

Nguồn: https://baotanghochiminh.vn/hien-vat-ke-chuyen.htm

**Quy trình**
1. Duyệt các trang danh sách (`/hien-vat-ke-chuyen/p-N.htm`) và lấy tiêu đề, link, ảnh đại diện, ngày đăng, lượt xem, tóm tắt.
2. (Tuỳ chọn) Vào từng bài chi tiết để lấy toàn văn và danh sách ảnh.
3. Lưu ra CSV / JSON / Excel.

> Lưu ý: chạy với tốc độ vừa phải (có `DELAY` giữa các request) và chỉ dùng dữ liệu theo đúng điều khoản của website.

In [1]:
!pip install -q requests beautifulsoup4 lxml pandas tqdm openpyxl

## 1. Cấu hình

In [2]:
import re, time, json, random
from pathlib import Path
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE      = "https://baotanghochiminh.vn"
LIST_URL  = BASE + "/hien-vat-ke-chuyen.htm"
PAGE_URL  = BASE + "/hien-vat-ke-chuyen/p-{n}.htm"

OUT_DIR   = Path("data_hien_vat_ke_chuyen")
OUT_DIR.mkdir(exist_ok=True)

FETCH_DETAIL    = True    # True: vào từng bài lấy toàn văn
DOWNLOAD_IMAGES = False   # True: tải ảnh đại diện + ảnh trong bài về máy
MAX_PAGES       = None    # None = tự phát hiện; hoặc đặt số, ví dụ 2 để test
DELAY           = (1.0, 2.0)  # nghỉ ngẫu nhiên (giây) giữa các request
CONTENT_SELECTOR = None   # nếu biết selector vùng nội dung bài, ví dụ "div.content-detail"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8",
})

def fetch(url, retries=3, timeout=30):
    for i in range(retries):
        try:
            r = session.get(url, timeout=timeout)
            if r.status_code == 200:
                r.encoding = "utf-8"
                return r.text
            if r.status_code == 404:
                return None
            print(f"[{r.status_code}] {url}")
        except requests.RequestException as e:
            print(f"Lỗi ({i+1}/{retries}) {url}: {e}")
        time.sleep(2 * (i + 1))
    return None

def polite_sleep():
    time.sleep(random.uniform(*DELAY))

## 2. Lấy danh sách bài từ các trang danh sách

In [3]:
DATE_RE  = re.compile(r"Ngày đăng\s*(\d{1,2}/\d{1,2}/\d{4})")
VIEWS_RE = re.compile(r"([\d\.,]+)\s*lượt xem")

def parse_list(html):
    soup = BeautifulSoup(html, "lxml")
    items, seen = [], set()

    # Mỗi bài có nút "Chi tiết" -> leo lên khối chứa ảnh + "Ngày đăng"
    for a in soup.find_all("a", string=lambda s: s and s.strip() == "Chi tiết"):
        url = urljoin(BASE, a.get("href", ""))
        if not url or url in seen:
            continue
        seen.add(url)

        block = a
        for _ in range(8):
            if block.parent is None:
                break
            block = block.parent
            if block.find("img") and "Ngày đăng" in block.get_text():
                break

        text  = block.get_text("\n", strip=True)
        title = (a.get("title") or "").strip()
        for link in block.find_all("a"):
            t = link.get_text(strip=True)
            if t and t != "Chi tiết":
                title = t
                break

        date_m, views_m = DATE_RE.search(text), VIEWS_RE.search(text)
        img = block.find("img")
        thumb = urljoin(BASE, img["src"]) if img and img.get("src") else None

        summary_lines = [
            ln for ln in text.split("\n")
            if ln != title and ln != "Chi tiết"
            and not DATE_RE.search(ln) and not VIEWS_RE.search(ln)
        ]
        items.append({
            "title": title,
            "url": url,
            "thumbnail": thumb,
            "date_posted": date_m.group(1) if date_m else None,
            "views": int(re.sub(r"[^\d]", "", views_m.group(1))) if views_m else None,
            "summary": " ".join(summary_lines).strip(),
        })
    return items, soup

def detect_last_page(soup):
    nums = [int(m.group(1)) for a in soup.find_all("a", href=True)
            if (m := re.search(r"/p-(\d+)\.htm", a["href"]))]
    return max(nums) if nums else 1

all_items = []
html = fetch(LIST_URL)
assert html, "Không tải được trang đầu. Kiểm tra kết nối mạng."
items, soup = parse_list(html)
last_page = detect_last_page(soup)
if MAX_PAGES:
    last_page = min(last_page, MAX_PAGES)
print(f"Số trang danh sách: {last_page}")
all_items.extend(items)

for n in tqdm(range(2, last_page + 1), desc="Trang danh sách"):
    polite_sleep()
    html = fetch(PAGE_URL.format(n=n))
    if not html:
        break
    items, _ = parse_list(html)
    if not items:
        break
    all_items.extend(items)

df = pd.DataFrame(all_items).drop_duplicates(subset="url").reset_index(drop=True)
print(f"Tổng số bài: {len(df)}")
df.head()

Số trang danh sách: 4


Trang danh sách:   0%|          | 0/3 [00:00<?, ?it/s]

Tổng số bài: 0


""


## 3. Vào từng bài chi tiết lấy toàn văn

In [4]:
NOISE_TAGS = ["script", "style", "nav", "header", "footer", "form", "iframe", "noscript"]

def find_content_node(soup):
    if CONTENT_SELECTOR:
        node = soup.select_one(CONTENT_SELECTOR)
        if node:
            return node
    # Heuristic: khối có nhiều chữ trong thẻ <p> nhất, chọn khối lồng sâu nhất
    scored = []
    for el in soup.find_all(["div", "article", "section"]):
        score = sum(len(p.get_text(strip=True)) for p in el.find_all("p"))
        if score:
            scored.append((score, el))
    if not scored:
        return soup.body
    top = max(s for s, _ in scored)
    cands = [el for s, el in scored if s >= 0.9 * top]
    return min(cands, key=lambda el: len(list(el.descendants)))

def parse_detail(html):
    soup = BeautifulSoup(html, "lxml")
    for t in soup(NOISE_TAGS):
        t.decompose()
    node = find_content_node(soup)

    paragraphs = [p.get_text(" ", strip=True) for p in node.find_all(["p", "h2", "h3", "h4", "li"])]
    paragraphs = [p for p in paragraphs if p]
    images = []
    for img in node.find_all("img"):
        src = img.get("src") or img.get("data-src")
        if src:
            images.append(urljoin(BASE, src))
    return {
        "content": "\n\n".join(paragraphs),
        "images": images,
    }

details = []
if FETCH_DETAIL:
    for url in tqdm(df["url"], desc="Bài chi tiết"):
        polite_sleep()
        html = fetch(url)
        details.append(parse_detail(html) if html else {"content": None, "images": []})
    df["content"] = [d["content"] for d in details]
    df["images"]  = [d["images"] for d in details]

df.head()

KeyError: 'url'

## 4. Kiểm tra nhanh chất lượng dữ liệu

In [ ]:
print(df.isna().sum())
if "content" in df:
    print("\nĐộ dài nội dung (ký tự):")
    print(df["content"].fillna("").str.len().describe())
    # In thử 1 bài
    row = df.iloc[0]
    print("\n---", row["title"], "---")
    print((row["content"] or "")[:800])

> Nếu nội dung bài bị lẫn menu / thiếu chữ: mở một trang chi tiết trong trình duyệt, bấm chuột phải -> Inspect vùng nội dung, rồi đặt `CONTENT_SELECTOR` ở ô cấu hình (ví dụ `"div.detail-content"`) và chạy lại mục 3.

## 5. (Tuỳ chọn) Tải ảnh

In [ ]:
def download(url, folder):
    folder.mkdir(parents=True, exist_ok=True)
    name = url.split("/")[-1].split("?")[0] or "image.jpg"
    path = folder / name
    if path.exists():
        return str(path)
    try:
        r = session.get(url, timeout=30)
        if r.status_code == 200:
            path.write_bytes(r.content)
            return str(path)
    except requests.RequestException as e:
        print("Lỗi tải ảnh:", url, e)
    return None

if DOWNLOAD_IMAGES:
    img_dir = OUT_DIR / "images"
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Tải ảnh"):
        slug = row["url"].rstrip("/").split("/")[-1].replace(".htm", "")
        paths = []
        urls = ([row["thumbnail"]] if row["thumbnail"] else []) + list(row.get("images", []) or [])
        for u in urls:
            p = download(u, img_dir / slug)
            if p:
                paths.append(p)
            time.sleep(0.3)
        df.at[i, "image_files"] = json.dumps(paths, ensure_ascii=False)

## 6. Lưu kết quả

In [ ]:
out = df.copy()
if "images" in out:
    out["images"] = out["images"].apply(lambda x: json.dumps(x, ensure_ascii=False))

out.to_csv(OUT_DIR / "hien_vat_ke_chuyen.csv", index=False, encoding="utf-8-sig")
out.to_excel(OUT_DIR / "hien_vat_ke_chuyen.xlsx", index=False)
df.to_json(OUT_DIR / "hien_vat_ke_chuyen.json", orient="records", force_ascii=False, indent=2)

print("Đã lưu vào:", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("hien_vat_ke_chuyen.*")):
    print(" -", f.name)

In [ ]:
# Nếu chạy trên Google Colab, bỏ comment để tải file về máy
# from google.colab import files
# files.download(str(OUT_DIR / "hien_vat_ke_chuyen.xlsx"))